# Notebook 04: Texas oil and gas well-site footprint

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

# I set up the folders for my wells analysis.
ROOT = Path(r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint")
INPUT = ROOT / "05_Optimized_Data" / "GeoParquet"
OUTPUT = ROOT / "06_Energy_Footprint_Analysis" / "Wells"
OUTPUT.mkdir(parents=True, exist_ok=True)

ACRE_M2 = 4046.8564224

# I load the cleaned spatial data and check that the geometries are valid for analysis.
def read_layer(name):
    data = gpd.read_parquet(INPUT / f"{name}.parquet")
    if data.crs is None or data.geometry.isna().any() or data.geometry.is_empty.any():
        raise ValueError(f"{name}: missing CRS or invalid geometry")
    return data

# I standardize county GEOIDs and flag invalid Texas county codes.
def county_key(df):
    codes = df["county_geoid"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    return codes.where(codes.str.fullmatch(r"48\d{3}"), pd.NA)

In [2]:

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

# I load the complete Texas well dataset.
wells = read_layer("Texas_Wells_All_Categories")

# I check the record count, required fields, and unique well IDs.
assert len(wells) == 1_380_352
assert {"source_row_id", "SYMNUM", "county_geoid"}.issubset(wells.columns)
assert wells["source_row_id"].notna().all() and wells["source_row_id"].is_unique

# I standardize county codes and well classifications.
wells["county_geoid"] = county_key(wells)
wells["symnum"] = pd.to_numeric(wells["SYMNUM"], errors="coerce").astype("Int64")

# I keep these classifications provisional until their definitions are verified.
HORIZONTAL_DIRECTIONAL_CODES = [86, 87]
PROVISIONAL_EXCLUDE_CODES = [2, 3, 7, 8, 9, 10]

# I summarize all well classifications before deciding which wells to include.
symnum_review = (
    wells.groupby("symnum", dropna=False)
    .agg(
        well_records=("source_row_id", "size"),
        counties=("county_geoid", "nunique")
    )
    .reset_index()
    .sort_values("well_records", ascending=False)
)

symnum_review["percent_of_records"] = (
    100 * symnum_review["well_records"] / len(wells)
).round(2)

# I flag classifications that need further review.
symnum_review["review_flag"] = "Review eligibility"

symnum_review.loc[
    symnum_review["symnum"].isin(HORIZONTAL_DIRECTIONAL_CODES),
    "review_flag"
] = "Review horizontal/directional classification"

symnum_review.loc[
    symnum_review["symnum"].isin(PROVISIONAL_EXCLUDE_CODES),
    "review_flag"
] = "Review proposed exclusion"

# I preserve the full dataset and create a provisional subset for diagnostics.
wells_all = wells.copy()

candidate_mask = (
    wells["symnum"].notna()
    & ~wells["symnum"].isin(PROVISIONAL_EXCLUDE_CODES)
)

wells_candidates = wells.loc[candidate_mask].copy()

wells_candidates["horizontal_directional_candidate"] = (
    wells_candidates["symnum"].isin(HORIZONTAL_DIRECTIONAL_CODES)
)

# I save the classification audit for data checking.
audit_output = OUTPUT / "Classification_Audit"
audit_output.mkdir(parents=True, exist_ok=True)

symnum_review.to_csv(
    audit_output / "well_symnum_classification_review.csv",
    index=False
)

symnum_review[["symnum", "well_records", "review_flag"]].to_csv(
    audit_output / "well_classification_rules_to_review.csv",
    index=False
)

# I report the classifications without assuming candidate wells are active.
print(f"Total well records: {len(wells_all):,}")
print(f"Dry-hole records (SYMNUM 3): {wells['symnum'].eq(3).sum():,}")
print(f"Distinct SYMNUM codes: {wells['symnum'].nunique()}")
print(f"Missing SYMNUM: {wells['symnum'].isna().sum():,}")
print(f"Provisional candidates: {len(wells_candidates):,}")
print(f"Outside candidate subset: {(~candidate_mask).sum():,}")
print(f"Horizontal/directional candidates: {wells_candidates['horizontal_directional_candidate'].sum():,}")

display(symnum_review)

Total well records: 1,380,352
Dry-hole records (SYMNUM 3): 288,905
Distinct SYMNUM codes: 54
Missing SYMNUM: 0
Provisional candidates: 554,478
Outside candidate subset: 825,874
Horizontal/directional candidates: 171,735


,symnum,well_records,counties,percent_of_records,review_flag
6,7,325237,227,23.56,Review proposed exclusion
2,3,288905,254,20.93,Review proposed exclusion
3,4,239950,210,17.38,Review eligibility
32,86,144958,208,10.50,Review horizontal/directional classification
1,2,88272,252,6.39,Review proposed exclusion
4,5,79108,209,5.73,Review eligibility
7,8,61541,209,4.46,Review proposed exclusion
8,9,39438,238,2.86,Review proposed exclusion
33,87,26777,204,1.94,Review horizontal/directional classification
17,21,24162,193,1.75,Review eligibility


In [3]:
# I inspect the well classifications before selecting records
# for the physical land footprint analysis.

classification_check = (
    wells.groupby("symnum", dropna=False)
    .agg(
        records=("source_row_id", "size"),
        counties=("county_geoid", "nunique")
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

# I identify classifications previously proposed for exclusion.
provisional_exclusions = [2, 3, 7, 8, 9, 10]

classification_check["proposed_exclusion"] = (
    classification_check["symnum"].isin(provisional_exclusions)
)

# I identify the classifications previously proposed
# as horizontal or directional wells.
classification_check["proposed_horizontal_directional"] = (
    classification_check["symnum"].isin([86, 87])
)

# I check the spatial reference and geometry before clustering.
print("Well CRS:", wells.crs)
print("CRS is projected:", wells.crs.is_projected)
print("CRS units:", wells.crs.axis_info[0].unit_name)
print("Missing geometries:", wells.geometry.isna().sum())
print("Empty geometries:", wells.geometry.is_empty.sum())

print("\nComplete SYMNUM classification inventory:")
display(classification_check)

Well CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Texas Centric Albers Equal Area", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "Texas Centric Albers Equal Area", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 18, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name": "Longitude of false origin", "value": -100, "unit": "degree", "id": 

,symnum,records,counties,proposed_exclusion,proposed_horizontal_directional
6,7,325237,227,True,False
2,3,288905,254,True,False
3,4,239950,210,False,False
32,86,144958,208,False,True
1,2,88272,252,True,False
4,5,79108,209,False,False
7,8,61541,209,True,False
8,9,39438,238,True,False
33,87,26777,204,False,True
17,21,24162,193,False,False


In [3]:
# I inspect the available fields to identify well classification information.
status_keywords = ["status", "type", "sym", "plug", "shut", "well", "description"]

status_columns = [
    col for col in wells.columns
    if any(word in col.lower() for word in status_keywords)
]

print("Potential well classification columns:")
print(status_columns)

# I examine the horizontal/directional codes flagged by the data checker.
print("\nHorizontal/directional well records:")
display(
    symnum_review[
        symnum_review["symnum"].isin([86, 87])
    ]
)

# I review all SYMNUM categories before finalizing exclusions.
display(symnum_review)

Potential well classification columns:
['WELLID', 'SYMNUM', 'symnum_normalized', 'well_category', 'county_match_status', 'well_scenario_label', 'symnum']

Horizontal/directional well records:


,symnum,well_records,counties,percent_of_records,review_flag
32,86,144958,208,10.50,Review horizontal/directional classification
33,87,26777,204,1.94,Review horizontal/directional classification


,symnum,well_records,counties,percent_of_records,review_flag
6,7,325237,227,23.56,Review proposed exclusion
2,3,288905,254,20.93,Review proposed exclusion
3,4,239950,210,17.38,Review eligibility
32,86,144958,208,10.50,Review horizontal/directional classification
1,2,88272,252,6.39,Review proposed exclusion
4,5,79108,209,5.73,Review eligibility
7,8,61541,209,4.46,Review proposed exclusion
8,9,39438,238,2.86,Review proposed exclusion
33,87,26777,204,1.94,Review horizontal/directional classification
17,21,24162,193,1.75,Review eligibility


In [4]:
# I compare SYMNUM codes with the existing well classifications.
classification_lookup = (
    wells.groupby(
        ["symnum", "well_category", "well_scenario_label", "legacy_excluded_category"],
        dropna=False
    )
    .size()
    .reset_index(name="well_records")
    .sort_values("well_records", ascending=False)
)

# I summarize the number of wells in each category.
category_summary = (
    wells.groupby(["well_category", "well_scenario_label"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

# I identify SYMNUM codes assigned to more than one category or scenario.
classification_variation = (
    classification_lookup.groupby("symnum", dropna=False)
    .agg(
        categories=("well_category", "nunique"),
        scenario_labels=("well_scenario_label", "nunique")
    )
    .reset_index()
)

multiple_labels = classification_variation[
    (classification_variation["categories"] > 1) |
    (classification_variation["scenario_labels"] > 1)
]

print("WELL CATEGORY SUMMARY")
display(category_summary)

print("SYMNUM CLASSIFICATION LOOKUP")
display(classification_lookup)

print("SYMNUM CODES WITH MULTIPLE LABELS")
display(multiple_labels)

# I examine codes 86 and 87 to investigate the horizontal/directional classification.
print("HORIZONTAL/DIRECTIONAL CLASSIFICATION REVIEW")
display(classification_lookup[classification_lookup["symnum"].isin([86, 87])])

# I save the classification lookup for data checking.
classification_lookup.to_csv(
    OUTPUT / "well_symnum_category_lookup.csv",
    index=False
)

WELL CATEGORY SUMMARY


,well_category,well_scenario_label,records
6,Plugged well,Excluded from legacy restricted scenario,409259
3,Oil/gas well,Included in legacy restricted scenario (not ve...,336468
2,Dry hole,Excluded from legacy restricted scenario,288905
4,Other well infrastructure,Included in legacy restricted scenario (not ve...,213268
5,Permitted location,Excluded from legacy restricted scenario,88272
0,Canceled location,Excluded from legacy restricted scenario,39438
7,Shut-in well,Included in legacy restricted scenario (not ve...,3711
1,Core test,Included in legacy restricted scenario (not ve...,763
8,Unknown,Included in legacy restricted scenario (not ve...,268


SYMNUM CLASSIFICATION LOOKUP


,symnum,well_category,well_scenario_label,legacy_excluded_category,well_records
6,7,Plugged well,Excluded from legacy restricted scenario,True,325237
2,3,Dry hole,Excluded from legacy restricted scenario,True,288905
3,4,Oil/gas well,Included in legacy restricted scenario (not ve...,False,239950
32,86,Other well infrastructure,Included in legacy restricted scenario (not ve...,False,144958
1,2,Permitted location,Excluded from legacy restricted scenario,True,88272
4,5,Oil/gas well,Included in legacy restricted scenario (not ve...,False,79108
7,8,Plugged well,Excluded from legacy restricted scenario,True,61541
8,9,Canceled location,Excluded from legacy restricted scenario,True,39438
33,87,Other well infrastructure,Included in legacy restricted scenario (not ve...,False,26777
17,21,Other well infrastructure,Included in legacy restricted scenario (not ve...,False,24162


SYMNUM CODES WITH MULTIPLE LABELS


,symnum,categories,scenario_labels


HORIZONTAL/DIRECTIONAL CLASSIFICATION REVIEW


,symnum,well_category,well_scenario_label,legacy_excluded_category,well_records
32,86,Other well infrastructure,Included in legacy restricted scenario (not ve...,False,144958
33,87,Other well infrastructure,Included in legacy restricted scenario (not ve...,False,26777


In [5]:
# I define which well categories belong in the primary analysis.
PRIMARY_CATEGORIES = ["Oil/gas well", "Shut-in well"]
HORIZONTAL_DIRECTIONAL_CODES = [86, 87]

EXCLUDED_CATEGORIES = [
    "Dry hole", "Plugged well", "Permitted location",
    "Canceled location", "Core test", "Unknown"
]

# I classify all wells without removing records from the master dataset.
wells["analysis_decision"] = "Review separately"

wells.loc[
    wells["well_category"].isin(EXCLUDED_CATEGORIES),
    "analysis_decision"
] = "Exclude from primary analysis"

wells.loc[
    wells["well_category"].isin(PRIMARY_CATEGORIES)
    | wells["symnum"].isin(HORIZONTAL_DIRECTIONAL_CODES),
    "analysis_decision"
] = "Include in primary analysis"

# I separate included, excluded, and review-only wells.
eligible = wells[wells["analysis_decision"] == "Include in primary analysis"].copy()
excluded = wells[wells["analysis_decision"] == "Exclude from primary analysis"].copy()
review = wells[wells["analysis_decision"] == "Review separately"].copy()

# I check the expected inventory and confirm that spatial grouping can use meters.
assert len(eligible) == 511_914
assert eligible.geom_type.eq("Point").all()
assert eligible.crs.is_projected
assert eligible.crs.axis_info[0].unit_name.lower() in ["metre", "meter"]

# I save the inclusion and exclusion decisions for data checking.
decision_summary = (
    wells.groupby(["well_category", "symnum", "analysis_decision"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

decision_summary.to_csv(OUTPUT / "well_eligibility_decisions.csv", index=False)

print(f"Original wells: {len(wells):,}")
print(f"Included: {len(eligible):,}")
print(f"Excluded: {len(excluded):,}")
print(f"Review separately: {len(review):,}")

display(decision_summary)

Original wells: 1,380,352
Included: 511,914
Excluded: 826,905
Review separately: 41,533


,well_category,symnum,analysis_decision,records
22,Plugged well,7,Exclude from primary analysis,325237
2,Dry hole,3,Exclude from primary analysis,288905
3,Oil/gas well,4,Include in primary analysis,239950
18,Other well infrastructure,86,Include in primary analysis,144958
21,Permitted location,2,Exclude from primary analysis,88272
4,Oil/gas well,5,Include in primary analysis,79108
23,Plugged well,8,Exclude from primary analysis,61541
0,Canceled location,9,Exclude from primary analysis,39438
19,Other well infrastructure,87,Include in primary analysis,26777
8,Other well infrastructure,21,Review separately,24162


In [6]:
# I use the eligibility decisions established earlier.
eligible = wells[
    wells["analysis_decision"] == "Include in primary analysis"
].copy()

eligible["county_geoid"] = county_key(eligible)
eligible["is_horizontal_directional"] = eligible["symnum"].isin(
    HORIZONTAL_DIRECTIONAL_CODES
)

# I check that the selected records are suitable for spatial analysis.
assert eligible["source_row_id"].is_unique
assert not eligible["well_category"].isin(["Dry hole", "Plugged well"]).any()
assert eligible.geom_type.eq("Point").all()
assert eligible.crs.is_projected
assert eligible.crs.axis_info[0].unit_name.lower() in ["metre", "meter"]

# I initially assign one site to each well record.
eligible["site_id"] = "well_" + eligible["source_row_id"].astype("string")

# I group nearby horizontal/directional wells into potential shared sites.
# I use 25 meters as an exploratory threshold
clustered = eligible[
    eligible["is_horizontal_directional"] & eligible["county_geoid"].notna()
]

for county_id, idx in clustered.groupby("county_geoid").groups.items():
    block = clustered.loc[idx]
    xy = np.column_stack([block.geometry.x, block.geometry.y])

    pairs = np.array(
        list(cKDTree(xy).query_pairs(r=25)),
        dtype=np.int64
    ).reshape(-1, 2)

    if len(pairs):
        edges = np.vstack([pairs, pairs[:, ::-1]])
        graph = coo_matrix(
            (np.ones(len(edges)), (edges[:, 0], edges[:, 1])),
            shape=(len(block), len(block))
        )
        _, labels = connected_components(graph, directed=False)
    else:
        labels = np.arange(len(block))

    # I use the smallest source ID in each group to identify the site.
    ids = block["source_row_id"].astype("string").to_numpy()
    cluster_ids = pd.Series(ids).groupby(labels).transform("min").to_numpy()

    eligible.loc[block.index, "site_id"] = (
        "hd_" + str(county_id) + "_" + cluster_ids
    )

#NEW-NY1
# I investigate whether conventional wells are also close enough to share sites.
# I keep their existing site assignments until this diagnostic is reviewed.
conventional = eligible[
    ~eligible["is_horizontal_directional"]
    & eligible["county_geoid"].notna()
]

nearby_conventional = []

for county_id, block in conventional.groupby("county_geoid"):
    if len(block) < 2:
        continue

    xy = np.column_stack([block.geometry.x, block.geometry.y])
    pairs = cKDTree(xy).query_pairs(r=25)

    if pairs:
        nearby_conventional.append({
            "county_geoid": county_id,
            "nearby_pairs": len(pairs),
            "wells_in_nearby_pairs": len(set().union(*pairs))
        })

conventional_review = pd.DataFrame(
    nearby_conventional,
    columns=["county_geoid", "nearby_pairs", "wells_in_nearby_pairs"]
)

conventional_review.to_csv(
    OUTPUT / "conventional_wells_25m_review.csv",
    index=False
)

# I retain representative site coordinates for regional classification.
eligible["site_x"] = eligible.geometry.x
eligible["site_y"] = eligible.geometry.y

site = (
    eligible.groupby("site_id", as_index=False)
    .agg(
        county_geoid=("county_geoid", "first"),
        well_records=("source_row_id", "size"),
        horizontal_directional=("is_horizontal_directional", "max"),
        site_x=("site_x", "mean"),
        site_y=("site_y", "mean")
    )
)

site_gdf = gpd.GeoDataFrame(
    site,
    geometry=gpd.points_from_xy(site["site_x"], site["site_y"]),
    crs=eligible.crs
)

# I summarize sites by county and retain missing county assignments.
county_sites = (
    site[site["county_geoid"].notna()]
    .groupby("county_geoid", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        horizontal_directional_sites=("horizontal_directional", "sum")
    )
)

unassigned_sites = site[site["county_geoid"].isna()].copy()

# I verify that no records were lost during grouping.
assert site["well_records"].sum() == len(eligible)
assert site["site_id"].is_unique
assert county_sites["potential_sites"].sum() + len(unassigned_sites) == len(site)

# I save the site inventory for the regional acreage calculations.
site.to_csv(OUTPUT / "wells_potential_sites_pre_regional.csv", index=False)

site_gdf.to_file(
    OUTPUT / "wells_potential_sites_pre_regional.gpkg",
    layer="potential_sites",
    driver="GPKG"
)

county_sites.to_csv(
    OUTPUT / "wells_county_site_counts_pre_regional.csv",
    index=False
)

unassigned_sites.to_csv(
    OUTPUT / "wells_unassigned_sites_pre_regional.csv",
    index=False
)

print(f"Included well records: {len(eligible):,}")
print(f"Potential development sites: {len(site):,}")
print(f"Horizontal/directional sites: {site['horizontal_directional'].sum():,}")
print(f"Multiwell sites: {(site['well_records'] > 1).sum():,}")
print(f"Records consolidated: {len(eligible) - len(site):,}")

print("\nCONVENTIONAL WELL PROXIMITY REVIEW")
print(f"Conventional records: {len(conventional):,}")
print(f"Nearby conventional pairs: {conventional_review['nearby_pairs'].sum():,}")
print(f"Conventional wells in nearby pairs: {conventional_review['wells_in_nearby_pairs'].sum():,}")

display(conventional_review.head())
display(site_gdf.head())

Included well records: 511,914
Potential development sites: 436,776
Horizontal/directional sites: 96,597
Multiwell sites: 31,779
Records consolidated: 75,138

CONVENTIONAL WELL PROXIMITY REVIEW
Conventional records: 340,129
Nearby conventional pairs: 3,302
Conventional wells in nearby pairs: 5,279


,county_geoid,nearby_pairs,wells_in_nearby_pairs
0,48001,67,117
1,48003,20,40
2,48009,74,148
3,48013,4,8
4,48015,5,10


,site_id,county_geoid,well_records,horizontal_directional,site_x,site_y,geometry
0,hd_48001_1004,48001,1,True,1.889289e+06,7.526561e+06,POINT (1889288.606 7526560.872)
1,hd_48001_101,48001,1,True,1.919583e+06,7.557039e+06,POINT (1919583.269 7557038.878)
2,hd_48001_1075,48001,1,True,1.898026e+06,7.531059e+06,POINT (1898025.904 7531058.533)
3,hd_48001_149,48001,1,True,1.909081e+06,7.553426e+06,POINT (1909081.296 7553425.989)
4,hd_48001_158,48001,1,True,1.906459e+06,7.550711e+06,POINT (1906459.244 7550710.827)


In [7]:
import requests
from shapely.geometry import shape

# I reuse the saved EIA shale-play boundaries to avoid downloading them again.
cache = OUTPUT / "eia_shale_play_boundaries.gpkg"

if cache.exists():
    plays_gdf = gpd.read_file(cache)
else:
    url = (
        "https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/"
        "TightOil_ShaleGas_Plays_Lower48_EIA/FeatureServer/0/query"
    )

    params = {
        "where": "1=1",
        "outFields": "FID,Shale_play,Basin",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "geojson"
    }

    # I download the boundaries only if the cached file is missing.
    response = requests.get(url, params=params, timeout=45)
    response.raise_for_status()
    data = response.json()

    if "error" in data or not data.get("features"):
        raise ValueError("EIA shale-play boundaries could not be retrieved")

    plays_gdf = gpd.GeoDataFrame(
        [f["properties"] for f in data["features"]],
        geometry=[shape(f["geometry"]) for f in data["features"]],
        crs="EPSG:4326"
    )

    plays_gdf.to_file(cache, layer="shale_plays", driver="GPKG")

# I check the required fields and match the site coordinate system.
assert not plays_gdf.empty
assert {"Basin", "Shale_play"}.issubset(plays_gdf.columns)

plays_gdf = plays_gdf.to_crs(site_gdf.crs)

print(f"EIA shale-play polygons: {len(plays_gdf):,}")
print(f"CRS: {plays_gdf.crs}")

EIA shale-play polygons: 50
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Texas Centric Albers Equal Area", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "Texas Centric Albers Equal Area", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 18, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name": "Longitude of false origin", "value": -100, "

In [8]:
# I select the four producing regions from the EIA shale-play boundaries.
region_definitions = {
    "Permian": plays_gdf["Basin"].eq("Permian"),
    "Eagle Ford": plays_gdf["Shale_play"].eq("Eagle Ford"),
    "Barnett": plays_gdf["Shale_play"].eq("Barnett"),
    "Haynesville": plays_gdf["Shale_play"].eq("Haynesville-Bossier")
}

# I combine polygons belonging to the same region.
region_geometries = {}

for region, mask in region_definitions.items():
    selected = plays_gdf.loc[mask]
    if selected.empty:
        raise ValueError(f"No polygons found for {region}")

    region_geometries[region] = selected.geometry.union_all()
    print(f"{region}: {len(selected)} polygons")

# I check whether any producing regions overlap.
regions = list(region_geometries)

overlap_df = pd.DataFrame([
    {
        "region_1": a,
        "region_2": b,
        "overlap_acres": region_geometries[a].intersection(
            region_geometries[b]
        ).area / ACRE_M2
    }
    for i, a in enumerate(regions)
    for b in regions[i + 1:]
])

display(overlap_df.sort_values("overlap_acres", ascending=False))

# I create one geographic record per region for the site classification.
regions_gdf = gpd.GeoDataFrame(
    {
        "region": regions,
        "geometry": [region_geometries[r] for r in regions]
    },
    crs=plays_gdf.crs
)

display(
    regions_gdf.assign(
        area_sq_km=regions_gdf.geometry.area / 1_000_000
    )[["region", "area_sq_km"]]
)

Permian: 7 polygons
Eagle Ford: 1 polygons
Barnett: 1 polygons
Haynesville: 1 polygons


,region_1,region_2,overlap_acres
0,Permian,Eagle Ford,0.0
1,Permian,Barnett,0.0
2,Permian,Haynesville,0.0
3,Eagle Ford,Barnett,0.0
4,Eagle Ford,Haynesville,0.0
5,Barnett,Haynesville,0.0


,region,area_sq_km
0,Permian,63400.136804
1,Eagle Ford,37930.673283
2,Barnett,68146.324654
3,Haynesville,28921.536468


In [10]:
# ASSIGN POTENTIAL WELL SITES TO PRODUCING REGIONS


# I assign each potential development site to one EIA-derived region.
# I use the representative site coordinates created after
# the 25-meter spatial grouping.

assert len(site_gdf) == 436_776
assert site_gdf["site_id"].is_unique
assert site_gdf.crs == regions_gdf.crs

# I keep only the fields needed for the spatial join.

region_boundaries = regions_gdf[
    ["region", "geometry"]
].copy()

# I identify which regional polygon contains each site point.
# I use intersects so points on polygon boundaries are included.

site_regions = gpd.sjoin(
    site_gdf,
    region_boundaries,
    how="left",
    predicate="intersects"
)

# I check whether any sites were assigned to multiple regions.
# I do not silently remove duplicate assignments.

duplicate_assignments = (
    site_regions["site_id"].duplicated().sum()
)

print(
    "Sites assigned more than once:",
    f"{duplicate_assignments:,}"
)

assert duplicate_assignments == 0, (
    "Some sites intersect multiple regions. "
    "Review boundary assignments before proceeding."
)

# I classify sites outside the selected polygons as Other Texas.

site_regions["region"] = (
    site_regions["region"]
    .fillna("Other Texas")
)

site_regions = site_regions.drop(
    columns=["index_right"]
).copy()

# I preserve the regional classifications in a GeoDataFrame.

site_regions = gpd.GeoDataFrame(
    site_regions,
    geometry="geometry",
    crs=site_gdf.crs
)

# VALIDATE REGIONAL CLASSIFICATION

# I verify that all potential sites and well records are retained.

assert len(site_regions) == len(site_gdf)
assert site_regions["site_id"].is_unique
assert site_regions["region"].notna().all()

assert (
    site_regions["well_records"].sum()
    == 511_914
)

# SUMMARIZE REGIONAL RESULTS

# I calculate the number of potential sites and well records
# assigned to each geographic region.

regional_summary = (
    site_regions.groupby("region", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        horizontal_directional_sites=(
            "horizontal_directional", "sum"
        )
    )
)

regional_summary["share_of_sites_pct"] = (
    regional_summary["potential_sites"]
    / len(site_regions)
    * 100
)

regional_summary = (
    regional_summary
    .sort_values(
        "potential_sites",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nREGIONAL WELL-SITE CLASSIFICATION")

display(regional_summary)

print("\nVALIDATION")

print(
    "Total potential sites:",
    f"{regional_summary['potential_sites'].sum():,}"
)

print(
    "Total well records:",
    f"{regional_summary['well_records'].sum():,}"
)

print(
    "Sites outside selected regions:",
    f"{(site_regions['region'] == 'Other Texas').sum():,}"
)

# SAVE REGIONAL SITE INVENTORY

# I save the geographic classifications before applying
# regional acreage assumptions.

site_regions.drop(
    columns="geometry"
).to_csv(
    OUTPUT / "wells_sites_by_region.csv",
    index=False
)

site_regions.to_file(
    OUTPUT / "wells_sites_by_region.gpkg",
    layer="well_sites",
    driver="GPKG"
)

regional_summary.to_csv(
    OUTPUT / "wells_regional_site_summary.csv",
    index=False
)

print("\nPASS: Regional classifications saved.")

Sites assigned more than once: 0

REGIONAL WELL-SITE CLASSIFICATION


,region,potential_sites,well_records,horizontal_directional_sites,share_of_sites_pct
0,Other Texas,254251,259423,38226,58.210845
1,Permian,65517,99528,23684,15.000137
2,Barnett,65434,76732,10582,14.981134
3,Eagle Ford,28458,50905,19515,6.515468
4,Haynesville,23116,25326,4590,5.292415



VALIDATION
Total potential sites: 436,776
Total well records: 511,914
Sites outside selected regions: 254,251

PASS: Regional classifications saved.


In [10]:
# I assign potential well sites to the EIA producing regions.
site_regions = site_gdf.copy()
site_regions["region"] = "Other Texas"

# I use a fixed priority if regional polygons overlap.
for region in ["Permian", "Eagle Ford", "Barnett", "Haynesville"]:
    inside = site_regions.geometry.intersects(region_geometries[region])
    site_regions.loc[
        inside & site_regions["region"].eq("Other Texas"),
        "region"
    ] = region

# I check how many potential sites fall within each region.
print("WELL SITES BY REGION")
display(site_regions["region"].value_counts())

# I confirm that all sites have a geographic classification.
assert site_regions["region"].notna().all()

WELL SITES BY REGION


region
Other Texas    254251
Permian         65517
Barnett         65434
Eagle Ford      28458
Haynesville     23116
Name: count, dtype: int64

In [11]:
# I examine well sites outside the selected EIA producing regions.
other_texas = site_regions[site_regions["region"] == "Other Texas"].copy()

# I summarize these sites by county to identify where they are concentrated.
other_counties = (
    other_texas.dropna(subset=["county_geoid"])
    .groupby("county_geoid", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        horizontal_directional_sites=("horizontal_directional", "sum")
    )
)

# I add county names when the reference table is available.
if "county_reference" in globals():
    other_counties = other_counties.merge(
        county_reference[["county_geoid", "county_name"]],
        on="county_geoid",
        how="left",
        validate="one_to_one"
    )
else:
    other_counties["county_name"] = pd.NA

# I calculate each county's share of sites classified as Other Texas.
other_counties["share_of_other_texas_pct"] = (
    100 * other_counties["potential_sites"] / len(other_texas)
)

other_counties = other_counties.sort_values(
    "potential_sites", ascending=False
).reset_index(drop=True)

print(f"Other Texas sites: {len(other_texas):,}")
print(f"Counties represented: {len(other_counties):,}")
print(f"Top 25 counties' share: {other_counties.head(25)['share_of_other_texas_pct'].sum():.1f}%")

display(other_counties.head(25))

# I save the county distribution for the geographic classification review.
other_counties.to_csv(
    OUTPUT / "wells_other_texas_county_review.csv",
    index=False
)

Other Texas sites: 254,251
Counties represented: 209
Top 25 counties' share: 49.1%


,county_geoid,potential_sites,well_records,horizontal_directional_sites,county_name,share_of_other_texas_pct
0,48485,13789,13789,3,<NA>,5.423381
1,48003,10037,10118,1510,<NA>,3.947674
2,48135,8075,8171,923,<NA>,3.175995
3,48435,6504,6514,131,<NA>,2.558102
4,48233,6032,6034,72,<NA>,2.372459
5,48103,6030,6341,551,<NA>,2.371672
6,48371,5836,5939,1303,<NA>,2.295370
7,48105,5158,5163,217,<NA>,2.028704
8,48501,4876,5100,1665,<NA>,1.917790
9,48179,4530,4530,12,<NA>,1.781704


In [12]:
# I reuse the existing county reference file instead of downloading it again.
county_source = OUTPUT / "wells_county_254_with_names.csv"
county_lookup = pd.read_csv(county_source, dtype={"county_geoid": "string"})

# I keep only county identifiers and names.
name_col = next(
    c for c in county_lookup.columns
    if c.lower() in ["county_name", "name"]
)

county_lookup = county_lookup[
    ["county_geoid", name_col]
].rename(columns={name_col: "county_name"})

county_lookup["county_geoid"] = county_key(county_lookup)

# I check that the reference contains unique Texas counties.
assert len(county_lookup) == 254
assert county_lookup["county_geoid"].is_unique

# I add county names to the Other Texas summary.
other_counties_named = other_counties.drop(
    columns="county_name", errors="ignore"
).merge(
    county_lookup,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

# I check that every county received a name.
assert other_counties_named["county_name"].notna().all()

display(other_counties_named.head(25))

# I save the named county summary for the geographic review.
other_counties_named.to_csv(
    OUTPUT / "wells_other_texas_counties_named.csv",
    index=False
)

,county_geoid,potential_sites,well_records,horizontal_directional_sites,share_of_other_texas_pct,county_name
0,48485,13789,13789,3,5.423381,Wichita County
1,48003,10037,10118,1510,3.947674,Andrews County
2,48135,8075,8171,923,3.175995,Ector County
3,48435,6504,6514,131,2.558102,Sutton County
4,48233,6032,6034,72,2.372459,Hutchinson County
5,48103,6030,6341,551,2.371672,Crane County
6,48371,5836,5939,1303,2.295370,Pecos County
7,48105,5158,5163,217,2.028704,Crockett County
8,48501,4876,5100,1665,1.917790,Yoakum County
9,48179,4530,4530,12,1.781704,Gray County


In [15]:
# I summarize the original regional classification before making changes.
regional_summary = (
    site_regions.groupby("region", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        horizontal_directional_sites=("horizontal_directional", "sum")
    )
)

# I calculate each region's share of all potential sites.
regional_summary["share_of_sites_pct"] = (
    100 * regional_summary["potential_sites"] / len(site_regions)
)

regional_summary = regional_summary.sort_values(
    "potential_sites", ascending=False
).reset_index(drop=True)

# I check that the summary includes all sites and well records.
assert regional_summary["potential_sites"].sum() == len(site_regions)
assert regional_summary["well_records"].sum() == site_regions["well_records"].sum()

display(regional_summary)

,region,potential_sites,well_records,horizontal_directional_sites,share_of_sites_pct
0,Other Texas,254251,259423,38226,58.210845
1,Permian,65517,99528,23684,15.000137
2,Barnett,65434,76732,10582,14.981134
3,Eagle Ford,28458,50905,19515,6.515468
4,Haynesville,23116,25326,4590,5.292415


In [23]:
# I save the preliminary regional classification before testing other basin boundaries.
checkpoint = OUTPUT / "regional_classification_checkpoint"
checkpoint.mkdir(parents=True, exist_ok=True)

site_regions.to_file(
    checkpoint / "preliminary_site_regions.gpkg",
    layer="sites",
    driver="GPKG"
)

regions_gdf.to_file(
    checkpoint / "preliminary_region_boundaries.gpkg",
    layer="regions",
    driver="GPKG"
)

regional_summary.to_csv(
    checkpoint / "preliminary_region_summary.csv",
    index=False
)

print(f"Checkpoint saved: {checkpoint}")
print(f"Sites preserved: {len(site_regions):,}")

Checkpoint saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Wells\regional_classification_checkpoint
Sites preserved: 436,776


In [24]:
# I explicitly locate the two basin shapefiles used in the revised model.
# I do not substitute other boundaries if these exact files are unavailable.
PERMIAN_NAME = "PermianBasin_Extent_201712.shp"
BASIN_NAME = "SedimentaryBasins_US_May2011_v2.shp"

def find_unique_shapefile(name):
    matches = list(ROOT.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {name} under {ROOT}; found {len(matches)}. "
            f"Matches: {matches[:10]}"
        )
    return matches[0]

permian_path = find_unique_shapefile(PERMIAN_NAME)
basins_path = find_unique_shapefile(BASIN_NAME)
print("Permian extent:", permian_path)
print("Sedimentary basins:", basins_path)

# I verify the broader EIA Permian Basin geometry before applying it.
permian_extent = gpd.read_file(permian_path)
assert permian_extent.crs is not None
assert len(permian_extent) == 1
assert permian_extent.geometry.is_valid.all()
permian_extent = permian_extent.to_crs(site_gdf.crs)
print("Permian features:", len(permian_extent))

Permian extent: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\PermianBasin_Boundary_Structural_Tectonic\PermianBasin_Extent_201712.shp
Sedimentary basins: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\SedimentaryBasins_US_EIA\SedimentaryBasins_US_May2011_v2.shp
Permian features: 1


In [25]:
# I compare my original Permian classification with the broader basin boundary.
assert permian_extent.crs == site_regions.crs
assert len(permian_extent) == 1

permian_polygon = permian_extent.geometry.iloc[0]

# I check which well sites fall inside the broader Permian Basin.
inside_permian = site_regions.geometry.intersects(permian_polygon)

# I compare the number of sites and wells in each region.
comparison = (
    site_regions.assign(inside_broader_permian=inside_permian)
    .groupby(["region", "inside_broader_permian"], as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum")
    )
)

display(comparison)

# I check how many sites might need a different regional classification.
other_to_permian = (
    site_regions["region"].eq("Other Texas") & inside_permian
).sum()

permian_outside = (
    site_regions["region"].eq("Permian") & ~inside_permian
).sum()

other_regions_inside = (
    site_regions["region"].isin(["Barnett", "Eagle Ford", "Haynesville"])
    & inside_permian
).sum()

print(f"Sites inside broader Permian Basin: {inside_permian.sum():,}")
print(f"Other Texas sites inside broader Permian: {other_to_permian:,}")
print(f"Original Permian sites outside broader boundary: {permian_outside:,}")
print(f"Other named-region sites inside broader Permian: {other_regions_inside:,}")

# I also compare the geographic coverage of the two Permian boundaries.
original_permian = regions_gdf.loc[
    regions_gdf["region"] == "Permian", "geometry"
].iloc[0]

print("\nBOUNDARY COMPARISON")
print(
    f"Area shared by both boundaries: "
    f"{original_permian.intersection(permian_polygon).area / 1_000_000:,.1f} km²"
)
print(
    f"Original Permian area outside broader boundary: "
    f"{original_permian.difference(permian_polygon).area / 1_000_000:,.1f} km²"
)

,region,inside_broader_permian,potential_sites,well_records
0,Barnett,False,65434,76732
1,Eagle Ford,False,28458,50905
2,Haynesville,False,23116,25326
3,Other Texas,False,173295,177323
4,Other Texas,True,80956,82100
5,Permian,True,65517,99528


Sites inside broader Permian Basin: 146,473
Other Texas sites inside broader Permian: 80,956
Original Permian sites outside broader boundary: 0
Other named-region sites inside broader Permian: 0

BOUNDARY COMPARISON
Area shared by both boundaries: 63,400.1 km²
Original Permian area outside broader boundary: 0.0 km²


In [26]:
# I keep the original classification so I can compare it with the revised one.
site_regions_revised = site_regions.copy()
site_regions_revised["original_region"] = site_regions_revised["region"]

# I use the broader Permian Basin boundary to update the regional classification.
inside_permian = site_regions_revised.geometry.intersects(permian_polygon)
site_regions_revised.loc[inside_permian, "region"] = "Permian"

# I check that no sites or well records were lost.
assert len(site_regions_revised) == len(site_regions)
assert site_regions_revised["site_id"].is_unique
assert site_regions_revised["well_records"].sum() == site_regions["well_records"].sum()

# I summarize the revised number of sites and wells in each region.
revised_summary = (
    site_regions_revised.groupby("region", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        horizontal_directional_sites=("horizontal_directional", "sum")
    )
)

revised_summary["share_of_sites_pct"] = (
    100 * revised_summary["potential_sites"] / len(site_regions_revised)
)

revised_summary = revised_summary.sort_values(
    "potential_sites", ascending=False
).reset_index(drop=True)

display(revised_summary)

# I check which regions the reclassified sites originally belonged to.
changed = site_regions_revised[
    site_regions_revised["original_region"] != site_regions_revised["region"]
]

print(f"Sites reclassified: {len(changed):,}")
display(changed.groupby(["original_region", "region"]).size().reset_index(name="sites"))

# I save the revised classification separately from the original.
site_regions_revised.to_file(
    OUTPUT / "wells_sites_by_region_revised_permian.gpkg",
    layer="well_sites",
    driver="GPKG"
)

revised_summary.to_csv(
    OUTPUT / "wells_regional_summary_revised_permian.csv",
    index=False
)

print("Revised Permian classification saved.")

,region,potential_sites,well_records,horizontal_directional_sites,share_of_sites_pct
0,Other Texas,173295,177323,28407,39.675944
1,Permian,146473,181628,33503,33.535039
2,Barnett,65434,76732,10582,14.981134
3,Eagle Ford,28458,50905,19515,6.515468
4,Haynesville,23116,25326,4590,5.292415


Sites reclassified: 80,956


,original_region,region,sites
0,Other Texas,Permian,80956


Revised Permian classification saved.


In [27]:
# I load the sedimentary basin boundaries to find the Fort Worth Basin.
basins_gdf = gpd.read_file(basins_path)

print(f"Total basin features: {len(basins_gdf):,}")
print("Original CRS:", basins_gdf.crs)

# I search the basin names to find the correct boundary.
matches = basins_gdf[
    basins_gdf["NAME"].str.contains(
        "FORT|WORTH|BARNETT", case=False, na=False
    )
].copy()

display(matches.drop(columns="geometry"))

# I check that I found one basin before using it.
if len(matches) != 1:
    raise ValueError(f"Expected one Fort Worth Basin polygon, found {len(matches)}")

# I use the same CRS as my well sites for the spatial comparison.
fort_worth_gdf = matches.to_crs(site_regions_revised.crs)

assert fort_worth_gdf.geometry.is_valid.all()
assert fort_worth_gdf.crs.is_projected

print("Features:", len(fort_worth_gdf))
print("CRS:", fort_worth_gdf.crs)
print(
    "Area (km²):",
    round(fort_worth_gdf.geometry.area.sum() / 1_000_000, 1)
)

Total basin features: 32
Original CRS: EPSG:4326


,NAME,Area_sq_km,Area_sq_mi
8,FORT WORTH,90528.158467,34953.117397


Features: 1
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Texas Centric Albers Equal Area", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "Texas Centric Albers Equal Area", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 18, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name": "Longitude of false origin", "value": -100, "unit": "degree",

In [28]:
# I compare the broader Fort Worth Basin with my original Barnett boundary.
fort_worth_polygon = fort_worth_gdf.geometry.iloc[0]
original_barnett = regions_gdf.loc[
    regions_gdf["region"] == "Barnett", "geometry"
].iloc[0]

# I check which well sites fall inside the Fort Worth Basin.
inside_fort_worth = site_regions_revised.geometry.intersects(fort_worth_polygon)

# I compare the current regional classifications with the broader boundary.
comparison_fw = (
    site_regions_revised.assign(inside_fort_worth=inside_fort_worth)
    .groupby(["region", "inside_fort_worth"], as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum")
    )
)

display(comparison_fw)

# I check how many sites could be affected by using the broader boundary.
other_to_fort_worth = (
    site_regions_revised["region"].eq("Other Texas") & inside_fort_worth
).sum()

barnett_outside = (
    site_regions_revised["region"].eq("Barnett") & ~inside_fort_worth
).sum()

other_regions_inside = (
    site_regions_revised["region"].isin(["Permian", "Eagle Ford", "Haynesville"])
    & inside_fort_worth
).sum()

print(f"Sites inside Fort Worth Basin: {inside_fort_worth.sum():,}")
print(f"Other Texas sites inside Fort Worth: {other_to_fort_worth:,}")
print(f"Barnett sites outside Fort Worth: {barnett_outside:,}")
print(f"Other named-region sites inside Fort Worth: {other_regions_inside:,}")

# I compare how much the two geographic boundaries overlap.
print("\nBOUNDARY COMPARISON")
print(
    f"Shared area: "
    f"{original_barnett.intersection(fort_worth_polygon).area / 1_000_000:,.1f} km²"
)
print(
    f"Barnett area outside Fort Worth: "
    f"{original_barnett.difference(fort_worth_polygon).area / 1_000_000:,.1f} km²"
)
print(
    f"Fort Worth area outside Barnett: "
    f"{fort_worth_polygon.difference(original_barnett).area / 1_000_000:,.1f} km²"
)

,region,inside_fort_worth,potential_sites,well_records
0,Barnett,True,65434,76732
1,Eagle Ford,False,28458,50905
2,Haynesville,False,23116,25326
3,Other Texas,False,149408,153393
4,Other Texas,True,23887,23930
5,Permian,False,146473,181628


Sites inside Fort Worth Basin: 89,321
Other Texas sites inside Fort Worth: 23,887
Barnett sites outside Fort Worth: 0
Other named-region sites inside Fort Worth: 0

BOUNDARY COMPARISON
Shared area: 68,146.3 km²
Barnett area outside Fort Worth: 0.0 km²
Fort Worth area outside Barnett: 22,381.8 km²


In [29]:
# I keep the previous classification so I can track any changes.
site_regions_basins = site_regions_revised.copy()
site_regions_basins["region_before_fort_worth"] = site_regions_basins["region"]

# I identify sites inside the broader Fort Worth Basin.
inside_fort_worth = site_regions_basins.geometry.intersects(fort_worth_polygon)

# I keep the Permian classification where the two basin boundaries overlap.
permian_overlap = inside_fort_worth & site_regions_basins["region"].eq("Permian")
fort_worth_assign = inside_fort_worth & ~permian_overlap

site_regions_basins.loc[fort_worth_assign, "region"] = "Fort Worth Basin"

print(f"Fort Worth sites overlapping Permian: {permian_overlap.sum():,}")

# I check that no sites or well records were lost.
assert len(site_regions_basins) == len(site_regions_revised)
assert site_regions_basins["site_id"].is_unique
assert site_regions_basins["well_records"].sum() == site_regions_revised["well_records"].sum()

# I summarize the updated regional classification.
basin_summary = (
    site_regions_basins.groupby("region", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        horizontal_directional_sites=("horizontal_directional", "sum")
    )
)

basin_summary["share_of_sites_pct"] = (
    100 * basin_summary["potential_sites"] / len(site_regions_basins)
)

basin_summary = basin_summary.sort_values(
    "potential_sites", ascending=False
).reset_index(drop=True)

display(basin_summary)

# I check which regions the reassigned sites originally belonged to.
changed = site_regions_basins[
    site_regions_basins["region"] != site_regions_basins["region_before_fort_worth"]
]

print(f"Sites reclassified: {len(changed):,}")

display(
    changed.groupby(["region_before_fort_worth", "region"])
    .size()
    .reset_index(name="sites")
)

# I save the updated classification without replacing the earlier results.
site_regions_basins.to_file(
    OUTPUT / "wells_sites_by_region_revised_basins.gpkg",
    layer="well_sites",
    driver="GPKG"
)

basin_summary.to_csv(
    OUTPUT / "wells_regional_summary_revised_basins.csv",
    index=False
)

print("Updated basin classification saved.")

Fort Worth sites overlapping Permian: 0


,region,potential_sites,well_records,horizontal_directional_sites,share_of_sites_pct
0,Other Texas,149408,153393,28190,34.207008
1,Permian,146473,181628,33503,33.535039
2,Fort Worth Basin,89321,100662,10799,20.450071
3,Eagle Ford,28458,50905,19515,6.515468
4,Haynesville,23116,25326,4590,5.292415


Sites reclassified: 89,321


,region_before_fort_worth,region,sites
0,Barnett,Fort Worth Basin,65434
1,Other Texas,Fort Worth Basin,23887


Updated basin classification saved.


In [30]:
# INVESTIGATE SITES REMAINING IN OTHER TEXAS

# I examine the geological distribution of sites that remain
# outside my four named regions before expanding the boundaries.

other_sites = site_regions_basins.loc[
    site_regions_basins["region"] == "Other Texas"
].copy()

# I project the nationwide sedimentary basin boundaries
# into the same CRS as my well-site data.

basins_projected = basins_gdf.to_crs(other_sites.crs)

# I retain only the basin name and geometry for this comparison.

basins_projected = basins_projected[
    ["NAME", "geometry"]
].copy()

# I identify which sedimentary basin polygons contain these sites.
# I use an intersects join so boundary points are included.

other_basin_matches = gpd.sjoin(
    other_sites[["site_id", "well_records", "geometry"]],
    basins_projected,
    how="left",
    predicate="intersects"
)

# I summarize unique sites within each basin.
# I keep unmatched sites as a separate category.

other_basin_matches["NAME"] = (
    other_basin_matches["NAME"].fillna("Outside mapped basins")
)

basin_distribution = (
    other_basin_matches
    .groupby("NAME", as_index=False)
    .agg(
        potential_sites=("site_id", "nunique"),
        well_records=("well_records", "sum")
    )
    .sort_values("potential_sites", ascending=False)
)

print("OTHER TEXAS: SEDIMENTARY BASIN DISTRIBUTION")
display(basin_distribution)

print("\nVALIDATION")

print(
    "Original Other Texas sites:",
    f"{len(other_sites):,}"
)

print(
    "Unique sites represented:",
    f"{other_basin_matches['site_id'].nunique():,}"
)

print(
    "Sites intersecting multiple basin polygons:",
    f"{(other_basin_matches.groupby('site_id').size() > 1).sum():,}"
)

OTHER TEXAS: SEDIMENTARY BASIN DISTRIBUTION


,NAME,potential_sites,well_records
7,WESTERN GULF,84213,86892
3,Outside mapped basins,32989,33108
6,TX-LA-MS SALT,18153,18626
0,ANADARKO,10831,11500
5,PERMIAN,1817,1835
4,PALO DURO,576,580
2,MARIETTA,508,509
1,ARDMORE,321,343



VALIDATION
Original Other Texas sites: 149,408
Unique sites represented: 149,408
Sites intersecting multiple basin polygons: 0


In [31]:
# EXAMINE WELL TYPES ACROSS REGIONS

# I compare horizontal/directional and other sites across
# my five regions to inform the surface-disturbance assumptions.

regional_site_types = (
    site_regions_basins
    .groupby(
        ["region", "horizontal_directional"],
        as_index=False
    )
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum")
    )
)

regional_site_types["site_type"] = (
    regional_site_types["horizontal_directional"]
    .map({
        True: "Horizontal/directional",
        False: "Other included wells"
    })
)

regional_site_types["share_within_region_pct"] = (
    regional_site_types["potential_sites"]
    / regional_site_types.groupby("region")["potential_sites"].transform("sum")
    * 100
)

display(
    regional_site_types[
        [
            "region",
            "site_type",
            "potential_sites",
            "well_records",
            "share_within_region_pct"
        ]
    ].sort_values(["region", "site_type"])
)

print("\nVALIDATION")

assert (
    regional_site_types["potential_sites"].sum()
    == 436_776
)

assert (
    regional_site_types["well_records"].sum()
    == 511_914
)

print("PASS: All sites and well records are accounted for.")

,region,site_type,potential_sites,well_records,share_within_region_pct
1,Eagle Ford,Horizontal/directional,19515,41962,68.574742
0,Eagle Ford,Other included wells,8943,8943,31.425258
3,Fort Worth Basin,Horizontal/directional,10799,22140,12.090102
2,Fort Worth Basin,Other included wells,78522,78522,87.909898
5,Haynesville,Horizontal/directional,4590,6800,19.856377
4,Haynesville,Other included wells,18526,18526,80.143623
7,Other Texas,Horizontal/directional,28190,32175,18.867798
6,Other Texas,Other included wells,121218,121218,81.132202
9,Permian,Horizontal/directional,33503,68658,22.873158
8,Permian,Other included wells,112970,112970,77.126842



VALIDATION
PASS: All sites and well records are accounted for.


In [32]:
# EXAMINE MULTI-WELL DEVELOPMENT SITES BY REGION

# I examine the number of well records associated with each
# potential site to understand how the 25-meter grouping
# affects the surface-disturbance model.

site_grouping_summary = (
    site_regions_basins
    .groupby(
        ["region", "horizontal_directional"],
        as_index=False
    )
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        single_well_sites=(
            "well_records",
            lambda x: (x == 1).sum()
        ),
        multi_well_sites=(
            "well_records",
            lambda x: (x > 1).sum()
        ),
        maximum_wells_per_site=("well_records", "max")
    )
)

site_grouping_summary["multi_well_share_pct"] = (
    site_grouping_summary["multi_well_sites"]
    / site_grouping_summary["potential_sites"]
    * 100
)

site_grouping_summary["average_wells_per_site"] = (
    site_grouping_summary["well_records"]
    / site_grouping_summary["potential_sites"]
)

site_grouping_summary["site_type"] = (
    site_grouping_summary["horizontal_directional"].map({
        True: "Horizontal/directional",
        False: "Other included wells"
    })
)

display(
    site_grouping_summary[
        [
            "region",
            "site_type",
            "potential_sites",
            "multi_well_sites",
            "multi_well_share_pct",
            "average_wells_per_site",
            "maximum_wells_per_site"
        ]
    ].sort_values(["region", "site_type"])
)

print("\nVALIDATION")

assert site_grouping_summary["potential_sites"].sum() == 436_776
assert site_grouping_summary["well_records"].sum() == 511_914

print(
    "Total multi-well sites:",
    f"{site_grouping_summary['multi_well_sites'].sum():,}"
)

print("PASS: Site and well totals reconcile.")

,region,site_type,potential_sites,multi_well_sites,multi_well_share_pct,average_wells_per_site,maximum_wells_per_site
1,Eagle Ford,Horizontal/directional,19515,9168,46.979247,2.150243,18
0,Eagle Ford,Other included wells,8943,0,0.000000,1.000000,1
3,Fort Worth Basin,Horizontal/directional,10799,4519,41.846467,2.050190,38
2,Fort Worth Basin,Other included wells,78522,0,0.000000,1.000000,1
5,Haynesville,Horizontal/directional,4590,1353,29.477124,1.481481,9
4,Haynesville,Other included wells,18526,0,0.000000,1.000000,1
7,Other Texas,Horizontal/directional,28190,2656,9.421781,1.141362,22
6,Other Texas,Other included wells,121218,0,0.000000,1.000000,1
9,Permian,Horizontal/directional,33503,14083,42.035042,2.049309,20
8,Permian,Other included wells,112970,0,0.000000,1.000000,1



VALIDATION
Total multi-well sites: 31,779
PASS: Site and well totals reconcile.


## Regional acreage factors and limitations

In [33]:
import pandas as pd

# I use Pierre et al. (2018), Table 3, printed page 814
# (PDF page 10), for the Permian and Eagle Ford mean pad areas.
# The study reports 3,760 m² for Permian and 16,970 m² for Eagle Ford.
# I convert square meters to acres using 1 acre = 4,046.8564224 m².
# Source: https://doi.org/10.1007/s00267-018-1000-2

SQM_PER_ACRE = 4046.8564224

# I use the Barnett-area study's 2.6-acre mean production-site
# footprint for Fort Worth Basin as a provisional transferred value.
# Source: https://doi.org/10.1016/j.landurbplan.2017.08.005
#In the section Gas well production site metrics

# I use the DOE 2009 Shale Gas Primer's illustrative pad sizes:
# 3.5 acres for horizontal development (Haynesville proxy)
# and 2.0 acres for vertical development (Other Texas proxy).
# These are not measured regional averages.
# Source:  https://scholarworks.sfasu.edu/spatialsci/12/

acreage_research = pd.DataFrame([
    {
        "region": "Permian",
        "central_acres": 3760 / SQM_PER_ACRE,
        "evidence_type": "Measured regional mean",
        "source": "Pierre et al. (2018)",
        "source_page": "Printed p. 814, Table 3"
    },
    {
        "region": "Eagle Ford",
        "central_acres": 16970 / SQM_PER_ACRE,
        "evidence_type": "Measured regional mean",
        "source": "Pierre et al. (2018)",
        "source_page": "Printed p. 814, Table 3"
    },
    {
        "region": "Fort Worth Basin",
        "central_acres": 2.6,
        "evidence_type": "Transferred local production-site mean",
        "source": "Barnett-area study (2017)",
        "source_page": "Exact page pending verification"
    },
    {
        "region": "Haynesville",
        "central_acres": 3.5,
        "evidence_type": "Illustrative horizontal-pad proxy",
        "source": "DOE Shale Gas Primer (2009)",
        "source_page": "Exact page pending verification"
    },
    {
        "region": "Other Texas",
        "central_acres": 2.0,
        "evidence_type": "Illustrative vertical-pad proxy",
        "source": "DOE Shale Gas Primer (2009)",
        "source_page": "Exact page pending verification"
    }
])

# I verify that each geographic region has one assumption
# before applying any acreage estimates to the site inventory.

assert acreage_research["region"].is_unique

assert set(acreage_research["region"]) == set(
    site_regions_basins["region"].unique()
)

display(acreage_research)

,region,central_acres,evidence_type,source,source_page
0,Permian,0.929116,Measured regional mean,Pierre et al. (2018),"Printed p. 814, Table 3"
1,Eagle Ford,4.193378,Measured regional mean,Pierre et al. (2018),"Printed p. 814, Table 3"
2,Fort Worth Basin,2.600000,Transferred local production-site mean,Barnett-area study (2017),Exact page pending verification
3,Haynesville,3.500000,Illustrative horizontal-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
4,Other Texas,2.000000,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification


In [34]:
# APPLY RESEARCH-BASED REGIONAL ACREAGE ASSUMPTIONS

# I apply the regional acreage estimates to potential development
# sites rather than individual well records to avoid assigning
# a separate full pad footprint to every grouped well.

well_site_acres = site_regions_basins.merge(
    acreage_research[
        [
            "region",
            "central_acres",
            "evidence_type",
            "source",
            "source_page"
        ]
    ],
    on="region",
    how="left",
    validate="many_to_one"
)

# I verify that every potential site received an acreage assumption.

assert len(well_site_acres) == 436_776
assert well_site_acres["site_id"].is_unique
assert well_site_acres["central_acres"].notna().all()
assert well_site_acres["well_records"].sum() == 511_914

# I calculate the estimated central footprint by region.

regional_footprint = (
    well_site_acres
    .groupby("region", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        central_acres_per_site=("central_acres", "first"),
        estimated_acres=("central_acres", "sum")
    )
    .sort_values("estimated_acres", ascending=False)
    .reset_index(drop=True)
)

regional_footprint["share_of_statewide_footprint_pct"] = (
    regional_footprint["estimated_acres"]
    / regional_footprint["estimated_acres"].sum()
    * 100
)

print("RESEARCH-BASED CENTRAL FOOTPRINT ESTIMATES")
display(regional_footprint)

print("\nSTATEWIDE CENTRAL ESTIMATE")

print(
    f"{regional_footprint['estimated_acres'].sum():,.0f} acres"
)

print("\nVALIDATION")

assert regional_footprint["potential_sites"].sum() == 436_776
assert regional_footprint["well_records"].sum() == 511_914

print("PASS: All sites and well records are accounted for.")

RESEARCH-BASED CENTRAL FOOTPRINT ESTIMATES


,region,potential_sites,well_records,central_acres_per_site,estimated_acres,share_of_statewide_footprint_pct
0,Other Texas,149408,153393,2.000000,298816.000000,34.450326
1,Fort Worth Basin,89321,100662,2.600000,232234.600000,26.774195
2,Permian,146473,181628,0.929116,136090.442189,15.689790
3,Eagle Ford,28458,50905,4.193378,119335.160330,13.758083
4,Haynesville,23116,25326,3.500000,80906.000000,9.327607



STATEWIDE CENTRAL ESTIMATE
867,382 acres

VALIDATION
PASS: All sites and well records are accounted for.


In [35]:
# EXPORT FINAL REGIONAL WELL FOOTPRINT ESTIMATES

# I export the research-based regional acreage model separately
# from my earlier exploratory low, central, and high scenarios.
# I preserve county-unassigned sites in the statewide total.

from pathlib import Path
import pandas as pd
import numpy as np

# I confirm that the final regional model is available.

assert len(well_site_acres) == 436_776
assert well_site_acres["site_id"].is_unique
assert well_site_acres["central_acres"].notna().all()
assert well_site_acres["well_records"].sum() == 511_914

# I standardize county identifiers before aggregation.

well_site_acres["county_geoid"] = (
    well_site_acres["county_geoid"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

valid_county = (
    well_site_acres["county_geoid"]
    .str.fullmatch(r"48\d{3}")
    .fillna(False)
)

well_site_acres.loc[
    ~valid_county, "county_geoid"
] = pd.NA

# I retain the site-level information needed for data checking.

site_export = well_site_acres[
    [
        "site_id",
        "county_geoid",
        "region",
        "well_records",
        "horizontal_directional",
        "central_acres",
        "evidence_type",
        "source",
        "source_page"
    ]
].copy()

site_export = site_export.rename(
    columns={"central_acres": "estimated_acres"}
)

# I calculate county acreage using only assigned sites.

county_regional = (
    site_export.loc[
        site_export["county_geoid"].notna()
    ]
    .groupby("county_geoid", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        central_acres=("estimated_acres", "sum")
    )
)

# I use the existing 254-county file as a geographic template.
# I retain county names without importing its old acreage.

county_template = pd.read_csv(
    OUTPUT / "wells_county_254_with_names.csv",
    dtype={"county_geoid": "string"}
)

county_template["county_geoid"] = (
    county_template["county_geoid"]
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

county_columns = [
    col for col in county_template.columns
    if col == "county_geoid" or (
        "county" in col.lower() and "acre" not in col.lower()
    )
]

county_template = county_template[
    county_columns
].drop_duplicates("county_geoid")

assert len(county_template) == 254
assert county_template["county_geoid"].is_unique

# I merge the regional acreage into all 254 Texas counties.

county_final = county_template.merge(
    county_regional,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

assert len(county_final) == 254

for col in ["potential_sites", "well_records", "central_acres"]:
    county_final[col] = county_final[col].fillna(0)

# I preserve sites without county assignments separately.

unassigned = site_export.loc[
    site_export["county_geoid"].isna()
].copy()

# I calculate the complete statewide footprint.

statewide_total = site_export["estimated_acres"].sum()
county_total = county_final["central_acres"].sum()
unassigned_total = unassigned["estimated_acres"].sum()

statewide_final = pd.DataFrame([{
    "potential_sites": len(site_export),
    "well_records": site_export["well_records"].sum(),
    "central_acres": statewide_total,
    "county_assigned_acres": county_total,
    "unassigned_acres": unassigned_total,
    "unassigned_sites": len(unassigned)
}])

# I verify that no site or acreage is lost during aggregation.

assert np.isclose(
    county_total + unassigned_total,
    statewide_total,
    atol=0.01,
    rtol=0
)

assert (
    county_final["potential_sites"].sum()
    + len(unassigned)
    == len(site_export)
)

assert (
    county_final["well_records"].sum()
    + unassigned["well_records"].sum()
    == 511_914
)

# I verify that county coverage is complete.

assert county_final["county_geoid"].notna().all()
assert county_final["county_geoid"].nunique() == 254
assert county_final["county_geoid"].is_unique

# I save the regional-model outputs under distinct filenames.

site_export.to_csv(
    OUTPUT / "wells_regional_site_acreage.csv",
    index=False
)

county_final.to_csv(
    OUTPUT / "wells_regional_county_254.csv",
    index=False
)

statewide_final.to_csv(
    OUTPUT / "wells_regional_statewide.csv",
    index=False
)

unassigned.to_csv(
    OUTPUT / "wells_regional_unassigned.csv",
    index=False
)

print("=" * 65)
print("FINAL REGIONAL WELL FOOTPRINT")
print("=" * 65)

print(f"Potential sites: {len(site_export):,}")
print(f"Well records: {site_export['well_records'].sum():,}")
print(f"County-assigned acres: {county_total:,.2f}")
print(f"Unassigned acres: {unassigned_total:,.2f}")
print(f"Statewide acres: {statewide_total:,.2f}")
print(f"Unassigned sites: {len(unassigned):,}")

print("\nFINAL STATEWIDE DATASET")
display(statewide_final)

print("\nFIRST FIVE COUNTY RECORDS")
display(county_final.head())

print("\nPASS: Regional acreage outputs saved and reconciled.")
# I document the precise regional acreage assumptions alongside final exports.
acreage_research.to_csv(OUTPUT / "wells_regional_acreage_assumptions.csv", index=False)
regional_footprint.to_csv(OUTPUT / "wells_regional_footprint_summary.csv", index=False)


FINAL REGIONAL WELL FOOTPRINT
Potential sites: 436,776
Well records: 511,914
County-assigned acres: 866,303.42
Unassigned acres: 1,078.79
Statewide acres: 867,382.20
Unassigned sites: 541

FINAL STATEWIDE DATASET


,potential_sites,well_records,central_acres,county_assigned_acres,unassigned_acres,unassigned_sites
0,436776,511914,867382.202519,866303.41517,1078.787349,541



FIRST FIVE COUNTY RECORDS


,county_geoid,county_name,potential_sites,well_records,central_acres
0,48001,Anderson County,1327.0,1348.0,2654.000000
1,48003,Andrews County,12981.0,14024.0,12060.857838
2,48005,Angelina County,190.0,263.0,560.000000
3,48007,Aransas County,451.0,484.0,902.000000
4,48009,Archer County,7471.0,7471.0,19424.600000



PASS: Regional acreage outputs saved and reconciled.


In [36]:

# I investigate the 541 sites without county assignments.
# I check whether they have usable coordinates or other
# geographic information that could help assign a county.

print("UNASSIGNED SITE DIAGNOSTIC")
print("=" * 65)

print("Total unassigned sites:", len(unassigned))
print("Total well records:", unassigned["well_records"].sum())
print("Total estimated acres:", unassigned["estimated_acres"].sum())

print("\nAvailable columns:")
print(unassigned.columns.tolist())

print("\nUnassigned sites by region:")
print(unassigned["region"].value_counts(dropna=False))

print("\nFirst 10 unassigned sites:")
display(unassigned.head(10))

# I check whether the original site-level dataset contains
# geographic fields that were excluded from the export.

print("\nGeographic fields in the original site dataset:")

geo_columns = [
    col for col in well_site_acres.columns
    if any(term in col.lower()
           for term in [
               "lat", "lon", "geometry",
               "county", "geoid", "x_coord", "y_coord"
           ])
]

print(geo_columns)

UNASSIGNED SITE DIAGNOSTIC
Total unassigned sites: 541
Total well records: 541
Total estimated acres: 1078.7873487029497

Available columns:
['site_id', 'county_geoid', 'region', 'well_records', 'horizontal_directional', 'estimated_acres', 'evidence_type', 'source', 'source_page']

Unassigned sites by region:
region
Other Texas    538
Permian          3
Name: count, dtype: int64

First 10 unassigned sites:


,site_id,county_geoid,region,well_records,horizontal_directional,estimated_acres,evidence_type,source,source_page
155072,well_121245,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155110,well_121267,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155117,well_121270,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155121,well_121271,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155424,well_121358,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155427,well_121359,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155433,well_121360,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155440,well_121361,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155442,well_121362,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
155446,well_121363,<NA>,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification



Geographic fields in the original site dataset:
['county_geoid', 'geometry']


In [37]:

import geopandas as gpd
import pandas as pd

# I inspect the original geometries for sites without
# county assignments before attempting a spatial correction.

missing_ids = set(unassigned["site_id"])

missing_sites = well_site_acres.loc[
    well_site_acres["site_id"].isin(missing_ids)
].copy()

print("Unassigned sites:", len(missing_sites))
print("Geometry column type:", missing_sites["geometry"].dtype)

print("\nFirst five geometries:")
display(
    missing_sites[
        ["site_id", "region", "geometry"]
    ].head()
)

print("\nMissing geometries:")
print(missing_sites["geometry"].isna().sum())

print("\nGeometry examples:")
for value in missing_sites["geometry"].dropna().head(3):
    print(type(value), str(value)[:200])

# I check whether the source is already a GeoDataFrame
# and identify its coordinate reference system.

print("\nOriginal dataset type:", type(well_site_acres))

if isinstance(well_site_acres, gpd.GeoDataFrame):
    print("CRS:", well_site_acres.crs)
else:
    print("CRS not attached to this DataFrame.")

Unassigned sites: 541
Geometry column type: geometry

First five geometries:


,site_id,region,geometry
155072,well_121245,Other Texas,POINT (1976286.521 7218354.077)
155110,well_121267,Other Texas,POINT (1971069.112 7210294.065)
155117,well_121270,Other Texas,POINT (1966573.273 7208362.438)
155121,well_121271,Other Texas,POINT (1967340.697 7207803.218)
155424,well_121358,Other Texas,POINT (1963000.292 7201520.434)



Missing geometries:
0

Geometry examples:
<class 'shapely.geometry.point.Point'> POINT (1976286.520748058 7218354.077379569)
<class 'shapely.geometry.point.Point'> POINT (1971069.1116908563 7210294.0653245365)
<class 'shapely.geometry.point.Point'> POINT (1966573.2729630326 7208362.438273535)

Original dataset type: <class 'geopandas.geodataframe.GeoDataFrame'>
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Texas Centric Albers Equal Area", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "c

In [38]:
# I load the Census county boundaries already saved in my project.
county_path = (
    ROOT
    / "01_Raw_Data"
    / "Texas_Boundaries"
    / "tl_2026_us_county"
    / "tl_2026_us_county.shp"
)

counties_all = gpd.read_file(county_path)

# I keep only Texas counties and match the well-site CRS.
counties_gdf = counties_all.loc[
    counties_all["STATEFP"].astype(str).str.zfill(2) == "48",
    ["GEOID", "NAME", "geometry"]
].copy()

counties_gdf = counties_gdf.to_crs(well_site_acres.crs)

print("Texas counties:", len(counties_gdf))
print("County CRS:", counties_gdf.crs)

assert len(counties_gdf) == 254

# I select sites that did not receive a county GEOID.
missing_sites = well_site_acres.loc[
    well_site_acres["county_geoid"].isna(),
    ["site_id", "geometry"]
].copy()

# I use a spatial intersection to check whether each site
# falls inside an official Texas county polygon.
matches = gpd.sjoin(
    missing_sites,
    counties_gdf[["GEOID", "NAME", "geometry"]],
    how="left",
    predicate="intersects"
)

# I count distinct county matches per site to identify
# unambiguous assignments and boundary cases.
match_counts = (
    matches.groupby("site_id")["GEOID"]
    .nunique()
)

print("\nCOUNTY MATCH RESULTS")
print("=" * 60)
print("Sites investigated:", len(missing_sites))
print("Matched to one county:", (match_counts == 1).sum())
print("Matched to multiple counties:", (match_counts > 1).sum())
print("Matched to no county:", (match_counts == 0).sum())

print("\nFirst 10 results:")
display(
    matches[
        ["site_id", "GEOID", "NAME"]
    ].head(10)
)

Texas counties: 254
County CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Texas Centric Albers Equal Area", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "Texas Centric Albers Equal Area", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 18, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name": "Longitude of false origin", "value": -100, "u

,site_id,GEOID,NAME
155072,well_121245,NaN,NaN
155110,well_121267,NaN,NaN
155117,well_121270,NaN,NaN
155121,well_121271,NaN,NaN
155424,well_121358,NaN,NaN
155427,well_121359,NaN,NaN
155433,well_121360,NaN,NaN
155440,well_121361,NaN,NaN
155442,well_121362,NaN,NaN
155446,well_121363,NaN,NaN


In [39]:

import pandas as pd
import numpy as np

# I select the 541 sites that could not be assigned
# to any Texas county.

missing_sites = well_site_acres.loc[
    well_site_acres["county_geoid"].isna(),
    ["site_id", "region", "central_acres", "geometry"]
].copy()

# I verify that both datasets use the same projected CRS.
assert missing_sites.crs == counties_gdf.crs

# I combine Texas county polygons to measure how far
# each unassigned site is from the state's mapped land area.
texas_boundary = counties_gdf.geometry.union_all()

missing_sites["distance_to_texas_m"] = (
    missing_sites.geometry.distance(texas_boundary)
)

print("DISTANCE FROM TEXAS COUNTY BOUNDARIES")
print("=" * 65)

print("\nDistance summary in meters:")
display(
    missing_sites["distance_to_texas_m"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

# I distinguish sites near county boundaries from
# sites located substantially farther away.

distance_bins = [0, 25, 100, 500, 1000, 5000, np.inf]

distance_labels = [
    "0–25 m",
    "25–100 m",
    "100–500 m",
    "500 m–1 km",
    "1–5 km",
    "More than 5 km"
]

missing_sites["distance_category"] = pd.cut(
    missing_sites["distance_to_texas_m"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True
)

print("\nSites by distance:")
print(
    missing_sites["distance_category"]
    .value_counts()
    .reindex(distance_labels, fill_value=0)
)

print("\nSites by region and distance:")
display(
    pd.crosstab(
        missing_sites["region"],
        missing_sites["distance_category"]
    )
)

print("\nTen sites farthest from Texas:")
display(
    missing_sites.nlargest(
        10, "distance_to_texas_m"
    )[
        [
            "site_id",
            "region",
            "central_acres",
            "distance_to_texas_m"
        ]
    ]
)

DISTANCE FROM TEXAS COUNTY BOUNDARIES

Distance summary in meters:


count      541.000000
mean      5898.857822
std       2849.330332
min          2.323356
25%       3616.794398
50%       6210.690441
75%       8203.102304
90%       9304.432845
95%      10152.907412
99%      10918.320766
max      11011.824234
Name: distance_to_texas_m, dtype: float64


Sites by distance:
distance_category
0–25 m              3
25–100 m            4
100–500 m          13
500 m–1 km         14
1–5 km            172
More than 5 km    335
Name: count, dtype: int64

Sites by region and distance:


distance_category,0–25 m,25–100 m,100–500 m,500 m–1 km,1–5 km,More than 5 km
region,,,,,,
Other Texas,1,3,13,14,172,335
Permian,2,1,0,0,0,0



Ten sites farthest from Texas:


,site_id,region,central_acres,distance_to_texas_m
375935,well_777678,Other Texas,2.0,11011.824234
179275,well_1292174,Other Texas,2.0,10977.814215
179260,well_1292137,Other Texas,2.0,10918.320766
179265,well_1292163,Other Texas,2.0,10918.320766
179266,well_1292164,Other Texas,2.0,10918.320766
179267,well_1292165,Other Texas,2.0,10918.320766
179268,well_1292166,Other Texas,2.0,10918.320766
179269,well_1292167,Other Texas,2.0,10918.320766
179270,well_1292168,Other Texas,2.0,10906.172685
375999,well_777895,Other Texas,2.0,10796.794979


In [40]:

import pandas as pd

# I convert unassigned site coordinates to latitude and
# longitude to identify where these locations actually fall.

missing_locations = missing_sites.to_crs("EPSG:4326").copy()

missing_locations["longitude"] = missing_locations.geometry.x
missing_locations["latitude"] = missing_locations.geometry.y

print("GEOGRAPHIC LOCATION CHECK")
print("=" * 65)

print("\nCoordinate summary:")
display(
    missing_locations[
        ["longitude", "latitude", "distance_to_texas_m"]
    ].describe()
)

# I inspect the most distant sites to determine whether
# they are near the Gulf Coast or neighboring states.

print("\n20 sites farthest from Texas:")
display(
    missing_locations.nlargest(
        20, "distance_to_texas_m"
    )[
        [
            "site_id",
            "region",
            "longitude",
            "latitude",
            "distance_to_texas_m"
        ]
    ]
)

print("\n20 sites nearest to Texas:")
display(
    missing_locations.nsmallest(
        20, "distance_to_texas_m"
    )[
        [
            "site_id",
            "region",
            "longitude",
            "latitude",
            "distance_to_texas_m"
        ]
    ]
)

GEOGRAPHIC LOCATION CHECK

Coordinate summary:


,longitude,latitude,distance_to_texas_m
count,541.000000,541.000000,541.000000
mean,-95.849853,28.478474,5898.857822
std,1.142303,0.841070,2849.330332
min,-103.413468,26.074110,2.323356
25%,-96.715774,28.105839,3616.794398
50%,-95.964583,28.489316,6210.690441
75%,-95.219669,28.889952,8203.102304
max,-93.797115,33.946715,11011.824234



20 sites farthest from Texas:


,site_id,region,longitude,latitude,distance_to_texas_m
375935,well_777678,Other Texas,-95.687212,28.555905,11011.824234
179275,well_1292174,Other Texas,-97.067848,26.463183,10977.814215
179260,well_1292137,Other Texas,-97.074189,26.484156,10918.320766
179265,well_1292163,Other Texas,-97.074189,26.484156,10918.320766
179266,well_1292164,Other Texas,-97.074189,26.484156,10918.320766
179267,well_1292165,Other Texas,-97.074189,26.484156,10918.320766
179268,well_1292166,Other Texas,-97.074189,26.484156,10918.320766
179269,well_1292167,Other Texas,-97.074189,26.484156,10918.320766
179270,well_1292168,Other Texas,-97.074310,26.484142,10906.172685
375999,well_777895,Other Texas,-96.086763,28.386208,10796.794979



20 sites nearest to Texas:


,site_id,region,longitude,latitude,distance_to_texas_m
321891,well_543684,Other Texas,-98.174962,26.074110,2.323356
217738,well_18592,Permian,-103.064562,32.171271,2.373296
228739,well_23019,Permian,-103.064588,32.163877,6.277623
366384,well_735446,Permian,-103.413468,32.000610,29.779703
389968,well_827749,Other Texas,-97.717711,33.946715,32.486509
263252,well_34910,Other Texas,-96.955343,27.879573,55.735425
286885,well_425208,Other Texas,-94.777577,29.228390,92.284763
262108,well_34542,Other Texas,-96.863956,27.979505,113.403275
161850,well_123582,Other Texas,-95.205577,28.925965,115.622542
403397,well_878055,Other Texas,-97.133953,27.621866,144.573628


In [41]:

import pandas as pd

# I group unassigned sites by their geographic coordinates
# to distinguish coastal concentrations from border locations.

locations = missing_locations.copy()

locations["longitude_group"] = (
    locations["longitude"].round(1)
)

locations["latitude_group"] = (
    locations["latitude"].round(1)
)

print("UNASSIGNED SITE GEOGRAPHIC CLUSTERS")
print("=" * 65)

clusters = (
    locations.groupby(
        ["longitude_group", "latitude_group"],
        dropna=False
    )
    .agg(
        sites=("site_id", "count"),
        acres=("central_acres", "sum"),
        mean_distance_m=("distance_to_texas_m", "mean")
    )
    .reset_index()
    .sort_values("sites", ascending=False)
)

display(clusters.head(20))

print("\nUnassigned sites by broad latitude range:")

locations["latitude_band"] = pd.cut(
    locations["latitude"],
    bins=[25, 27, 29, 31, 33, 35, 37],
    include_lowest=True
)

display(
    locations.groupby(
        "latitude_band",
        observed=True
    )
    .agg(
        sites=("site_id", "count"),
        acres=("central_acres", "sum")
    )
)

UNASSIGNED SITE GEOGRAPHIC CLUSTERS


,longitude_group,latitude_group,sites,acres,mean_distance_m
31,-96.2,28.3,29,58.0,9054.830246
9,-97.2,27.5,27,54.0,2241.287565
33,-96.1,28.4,25,50.0,8029.816559
11,-97.1,26.5,23,46.0,9808.107920
38,-95.8,28.5,22,44.0,7533.205249
36,-96.0,28.5,22,44.0,5342.546517
70,-94.1,29.5,20,40.0,6601.681353
32,-96.2,28.4,18,36.0,6491.809903
44,-95.5,28.7,17,34.0,5585.579262
51,-95.2,28.8,16,32.0,5788.620970



Unassigned sites by broad latitude range:


,sites,acres
latitude_band,,
"(24.999, 27.0]",29,58.000000
"(27.0, 29.0]",387,774.000000
"(29.0, 31.0]",121,242.000000
"(31.0, 33.0]",3,2.787349
"(33.0, 35.0]",1,2.000000


In [42]:
# I identified 541 eligible well sites that could not be assigned to a Texas
# county. Most are concentrated along the Gulf Coast and appear to be offshore,
# while four are near Texas's northern or western borders.
# Because this analysis measures onshore land footprints, I separate these
# sites from the provisional county-assigned acreage estimate.
# I retain all 541 records in an audit file rather than deleting them,
# since their geographic status has not been fully verified.
# This does not change my well eligibility criteria: I continue to include
# all well types except dry holes and plugged wells.

In [43]:
# I check which files the well footprint analysis saved
# so I can identify the final outputs and avoid keeping unnecessary files.

for file in sorted(ROOT.rglob("*")):
    if file.is_file() and (
        "well" in file.name.lower()
        or "footprint" in file.name.lower()
        or "unassigned" in file.name.lower()
    ):
        print(file.relative_to(ROOT))

01_Raw_Data\PRC-underground location\RRC-UIC_well_location_20261023.csv
01_Raw_Data\RRC_Wells\documents_20261015\well001.zip
01_Raw_Data\RRC_Wells\documents_20261015\well003.zip
01_Raw_Data\RRC_Wells\documents_20261015\well005.zip
01_Raw_Data\RRC_Wells\documents_20261015\well007.zip
01_Raw_Data\RRC_Wells\documents_20261015\well009.zip
01_Raw_Data\RRC_Wells\documents_20261015\well011.zip
01_Raw_Data\RRC_Wells\documents_20261015\well013.zip
01_Raw_Data\RRC_Wells\documents_20261015\well015.zip
01_Raw_Data\RRC_Wells\documents_20261015\well017.zip
01_Raw_Data\RRC_Wells\documents_20261015\well019.zip
01_Raw_Data\RRC_Wells\documents_20261015\well021.zip
01_Raw_Data\RRC_Wells\documents_20261015\well023.zip
01_Raw_Data\RRC_Wells\documents_20261015\well025.zip
01_Raw_Data\RRC_Wells\documents_20261015\well027.zip
01_Raw_Data\RRC_Wells\documents_20261015\well029.zip
01_Raw_Data\RRC_Wells\documents_20261015\well031.zip
01_Raw_Data\RRC_Wells\documents_20261015\well033.zip
01_Raw_Data\RRC_Wells\docum

In [44]:
import pandas as pd
from pathlib import Path

# I check that the final regional well footprint files exist
# and contain consistent site, county, and statewide results.

output_dir = ROOT / "06_Energy_Footprint_Analysis" / "Wells"

files_to_check = [
    "wells_regional_acreage_assumptions.csv",
    "wells_regional_site_acreage.csv",
    "wells_regional_county_254.csv",
    "wells_regional_statewide.csv",
    "wells_regional_site_summary.csv",
    "wells_regional_unassigned.csv",
    "wells_unassigned_county_audit.csv"
]

print("FILE EXISTENCE CHECK")
print("=" * 65)

for filename in files_to_check:
    path = output_dir / filename
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {filename}")

# I inspect the saved columns and row counts to confirm
# that these files contain the results I expect.

print("\nSAVED FILE CONTENTS")
print("=" * 65)

for filename in [
    "wells_regional_site_acreage.csv",
    "wells_regional_county_254.csv",
    "wells_regional_statewide.csv",
    "wells_regional_unassigned.csv"
]:
    path = output_dir / filename

    if path.is_file():
        df = pd.read_csv(path)
        print(f"\n{filename}")
        print(f"Rows: {len(df):,}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))

# I verify that the county-assigned and unassigned acreage
# reconcile with the current site-level analysis.

print("\nCURRENT NOTEBOOK RECONCILIATION")
print("=" * 65)

assigned = well_site_acres.loc[
    well_site_acres["county_geoid"].notna()
]
unassigned = well_site_acres.loc[
    well_site_acres["county_geoid"].isna()
]

print(f"Total potential sites: {len(well_site_acres):,}")
print(f"County-assigned sites: {len(assigned):,}")
print(f"Unassigned sites: {len(unassigned):,}")
print(f"County-assigned acres: {assigned['central_acres'].sum():,.2f}")
print(f"Unassigned acres: {unassigned['central_acres'].sum():,.2f}")
print(f"Combined acres: {well_site_acres['central_acres'].sum():,.2f}")

FILE EXISTENCE CHECK
FOUND: wells_regional_acreage_assumptions.csv
FOUND: wells_regional_site_acreage.csv
FOUND: wells_regional_county_254.csv
FOUND: wells_regional_statewide.csv
FOUND: wells_regional_site_summary.csv
FOUND: wells_regional_unassigned.csv
FOUND: wells_unassigned_county_audit.csv

SAVED FILE CONTENTS

wells_regional_site_acreage.csv
Rows: 436,776
Columns: ['site_id', 'county_geoid', 'region', 'well_records', 'horizontal_directional', 'estimated_acres', 'evidence_type', 'source', 'source_page']


,site_id,county_geoid,region,well_records,horizontal_directional,estimated_acres,evidence_type,source,source_page
0,hd_48001_1004,48001.0,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
1,hd_48001_101,48001.0,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
2,hd_48001_1075,48001.0,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification



wells_regional_county_254.csv
Rows: 254
Columns: ['county_geoid', 'county_name', 'potential_sites', 'well_records', 'central_acres']


,county_geoid,county_name,potential_sites,well_records,central_acres
0,48001,Anderson County,1327.0,1348.0,2654.000000
1,48003,Andrews County,12981.0,14024.0,12060.857838
2,48005,Angelina County,190.0,263.0,560.000000



wells_regional_statewide.csv
Rows: 1
Columns: ['potential_sites', 'well_records', 'central_acres', 'county_assigned_acres', 'unassigned_acres', 'unassigned_sites']


,potential_sites,well_records,central_acres,county_assigned_acres,unassigned_acres,unassigned_sites
0,436776,511914,867382.202519,866303.41517,1078.787349,541



wells_regional_unassigned.csv
Rows: 541
Columns: ['site_id', 'county_geoid', 'region', 'well_records', 'horizontal_directional', 'estimated_acres', 'evidence_type', 'source', 'source_page']


,site_id,county_geoid,region,well_records,horizontal_directional,estimated_acres,evidence_type,source,source_page
0,well_121245,NaN,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
1,well_121267,NaN,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification
2,well_121270,NaN,Other Texas,1,True,2.0,Illustrative vertical-pad proxy,DOE Shale Gas Primer (2009),Exact page pending verification



CURRENT NOTEBOOK RECONCILIATION
Total potential sites: 436,776
County-assigned sites: 436,235
Unassigned sites: 541
County-assigned acres: 866,303.42
Unassigned acres: 1,078.79
Combined acres: 867,382.20


In [45]:
# NEW-NY
# Final notes based on Tony's feedback

#Based on Tony's first round of comments, I reviewed the well-pad acreage assumptions and regional boundaries. I will make the following updates before finalizing the wells analysis.

#**1. Haynesville**
#I will replace the current 3.5 acres per site with 1.2 acres to use a conservative estimate based on Dans et al. (2012).
#Source: https://scholarworks.sfasu.edu/spatialsci/12/
#**2. Other Texas**
#Tony suggested using the overall average from Pierre et al. (2018). I checked the study, but it does not provide an overall average for the entire study area.
#I then found another source from the BLM environmental analysis. :https://www.bia.gov/sites/default/files/media_document/okt_final_joint_eis%2C_blm_rmp_and_bia_integrated_rmp_vol_4_508_compliant_v2.pdf
#On page 56, the report provides well-pad sizes based on 100 APD well samples from 2012 to 2014.
#To keep my estimate conservative, I will use:
#- 0.9 acres per site for vertical wells.
#- 1.2 acres per site for horizontal wells.

In [46]:
# NEW-NY
# I update the wells acreage estimates based on Tony's feedback.
# I keep my original results and save the revised estimates separately.

# I start with the regional classifications I already updated.
revised_wells = site_regions_basins.copy()

# I keep the existing acreage assumptions for these three regions.
regional_acres = {
    "Permian": 3760 / 4046.8564224,
    "Eagle Ford": 16970 / 4046.8564224,
    "Fort Worth Basin": 2.6,
    "Haynesville": 1.2
}

revised_wells["estimated_acres"] = (
    revised_wells["region"].map(regional_acres)
)

# I use the BLM minimum pad sizes for Other Texas.
# Source: BLM environmental analysis, page 56.

other = revised_wells["region"].eq("Other Texas")

revised_wells.loc[
    other & ~revised_wells["horizontal_directional"].fillna(False),
    "estimated_acres"
] = 0.9

revised_wells.loc[
    other & revised_wells["horizontal_directional"].fillna(False),
    "estimated_acres"
] = 1.2

# I check that all sites have an acreage estimate.
assert revised_wells["estimated_acres"].notna().all()
assert len(revised_wells) == len(site_regions_basins)
assert revised_wells["site_id"].is_unique
assert (
    revised_wells["well_records"].sum()
    == site_regions_basins["well_records"].sum()
)

# I summarize the updated acreage by region.
revised_footprint = (
    revised_wells.groupby("region", as_index=False)
    .agg(
        potential_sites=("site_id", "size"),
        well_records=("well_records", "sum"),
        estimated_acres=("estimated_acres", "sum")
    )
    .sort_values("estimated_acres", ascending=False)
    .reset_index(drop=True)
)

# I calculate each region's share of the statewide footprint.
revised_footprint["share_of_total_pct"] = (
    100 * revised_footprint["estimated_acres"]
    / revised_footprint["estimated_acres"].sum()
)

display(revised_footprint)

print(
    "Revised statewide wells footprint:",
    f"{revised_wells['estimated_acres'].sum():,.0f} acres"
)

# I compare the revised estimate with my earlier calculation.
original_total = well_site_acres["central_acres"].sum()
revised_total = revised_wells["estimated_acres"].sum()

comparison = pd.DataFrame({
    "scenario": ["Original", "Revised conservative"],
    "estimated_acres": [original_total, revised_total]
})

display(comparison)

print(
    "Change from original:",
    f"{revised_total - original_total:,.0f} acres"
)

# I save the revised results without overwriting earlier files.
revised_wells.to_file(
    OUTPUT / "wells_sites_revised_tony.gpkg",
    layer="well_sites",
    driver="GPKG"
)

revised_footprint.to_csv(
    OUTPUT / "wells_regional_footprint_revised_tony.csv",
    index=False
)

# I also save a site-level CSV for checking the acreage assumptions.
revised_wells[
    [
        "site_id",
        "county_geoid",
        "region",
        "well_records",
        "horizontal_directional",
        "estimated_acres"
    ]
].to_csv(
    OUTPUT / "wells_site_acreage_revised_tony.csv",
    index=False
)

print("Revised wells results saved successfully.")

,region,potential_sites,well_records,estimated_acres,share_of_total_pct
0,Fort Worth Basin,89321,100662,232234.600000,35.276663
1,Other Texas,149408,153393,142924.200000,21.710326
2,Permian,146473,181628,136090.442189,20.672271
3,Eagle Ford,28458,50905,119335.160330,18.127128
4,Haynesville,23116,25326,27739.200000,4.213612


Revised statewide wells footprint: 658,324 acres


,scenario,estimated_acres
0,Original,867382.202519
1,Revised conservative,658323.602519


Change from original: -209,059 acres
Revised wells results saved successfully.
